# Refined tin: price volatility and U.S. import dependence, 2021–2025
RQ: How did price volatility and U.S. import dependence change for refined tin, an input used in semiconductor packaging, from 2021 to 2025?

Run All. Uses three immutable input snapshots in `inputs/` and the Python standard library. Outputs go beside the notebook. Primary output: `tin_annual_combined_2021_2025.csv` (five annual observations). Monthly companion: `tin_monthly_combined_2021_2025.csv` (60 months).

**Definitions and limits:** realized annual volatility = sample standard deviation of the 12 monthly simple percentage price changes (not annualized). The January change uses the prior December price. Price is a tin benchmark in the supplied file, not a verified semiconductor-packaging-grade U.S. price. Official net import reliance is supplied only for 2025 (77%). Do not invent 2021–2024 official values. A separately labelled proxy = 100 × (refined imports − refined exports) / apparent refined consumption; it is NOT the USGS reliance calculation and does not reconcile to 77%. Annual supply-demand data are historical descriptions, not monthly values or information available in real time. 2025 quantities may be estimates in MCS2026; the supplied processed file does not preserve that flag.

Supplier HHI bounds describe 2021–2024 combined and are stored once in a separate context table, not repeated as annual values. Broad chapter-26 Comtrade trade is excluded because it is not refined-tin trade. Exact selection prevents accidentally matching platinum.

In [1]:
from pathlib import Path
import csv, math, statistics, json, hashlib
from collections import Counter
from datetime import date
START=Path.cwd().resolve()
BASE=next((p for p in (START,*START.parents) if (p/'inputs/minerals_supply_demand.csv').exists()),None)
if BASE is None:
    candidate=START/'Project_Directory/Final_data/final dataset'
    if (candidate/'inputs').exists(): BASE=candidate
if BASE is None: raise FileNotFoundError('Run from the notebook folder or repository root.')
def read(name):
    with (BASE/'inputs'/name).open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def write(name,rows,fields=None):
    with (BASE/name).open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields or list(rows[0]));w.writeheader();w.writerows(rows)
def num(v):
    try:x=float(str(v).replace(',',''))
    except (ValueError,TypeError):return None
    return x if math.isfinite(x) else None
supply=read('minerals_supply_demand.csv')
prices=read('tin_monthly_volatility.csv')
trade=read('final_trade_indicators.csv')
prices.sort(key=lambda r:r['month'])
assert len({r['month'] for r in prices})==len(prices), 'Duplicate price dates'
for r in prices:
    date.fromisoformat(r['month']); assert num(r['tin_price_usd_per_kg']) is not None and num(r['tin_price_usd_per_kg'])>0
# Recompute returns from the full price history before selecting 2021–2025.
returns={}
for a,b in zip(prices,prices[1:]):
    da,db=date.fromisoformat(a['month']),date.fromisoformat(b['month'])
    if db.year*12+db.month != da.year*12+da.month+1:raise ValueError('Gap in monthly prices')
    value=100*(num(b['tin_price_usd_per_kg'])/num(a['tin_price_usd_per_kg'])-1)
    supplied=num(b['monthly_price_change_pct'])
    if supplied is not None:assert abs(value-supplied)<1e-5,'Supplied monthly change mismatch'
    returns[b['month']]=value
selected=[r for r in prices if 2021<=int(r['month'][:4])<=2025]
assert len(selected)==60
# Use exact detail, because imports were misclassified under metric=consumption.
def annual_quantity(year,detail):
    rows=[r for r in supply if r['mineral'].strip().casefold()=='tin' and r['country']=='United States'
          and int(r['year'])==year and r['source']=='MCS2026' and r['detail'].strip()==detail]
    assert len(rows)==1,(year,detail,len(rows))
    r=rows[0];assert r['unit_raw'].strip()=='metric tons'
    value=num(r['value_t']);assert value is not None and value>=0
    return value
# Trade CSV can use either older or newer output schema.
tin_trade=[r for r in trade if r['reporter_country']=='United States'
           and r['commodity'].strip().casefold()=='tin, refined']
official={}
for r in tin_trade:
    if r['source']=='USGS MCS2026 Fig2' and r['indicator']=='import_reliance_exact' and num(r['value']) is not None:
        assert r['period_start']==r['period_end']
        y=int(r['period_start']);assert y not in official
        official[y]=num(r['value'])
print('Input row counts:',len(supply),len(prices),len(trade))
print('Official reliance years:',sorted(official))

Input row counts: 3831 802 1116
Official reliance years: [2025]


## Build the annual and monthly tables
Missing official reliance is written as `not_provided_in_inputs` with a status, not 0 or an imputed percentage. All calculated price and quantity measures must be complete. Monthly rows repeat annual quantities for descriptive joining only; they do not represent monthly imports and are not 60 independent annual dependence observations.

In [2]:
annual=[]
for year in range(2021,2026):
    months=[r for r in selected if int(r['month'][:4])==year]
    assert len(months)==12
    ret=[returns[r['month']] for r in months]
    imp=annual_quantity(year,'Imports for consumption: Refined')
    exp=annual_quantity(year,'Exports: Refined')
    cons=annual_quantity(year,'Consumption, apparent, refined');assert cons>0
    annual.append(dict(year=year,mineral='tin',material_scope='refined tin',reporter_country='United States',
        price_month_count=12,mean_tin_price_usd_per_kg=statistics.mean(num(r['tin_price_usd_per_kg']) for r in months),
        realized_monthly_return_volatility_pct=statistics.stdev(ret),
        refined_imports_tonnes=imp,refined_exports_tonnes=exp,apparent_refined_consumption_tonnes=cons,
        refined_net_imports_tonnes=imp-exp,net_imports_to_consumption_proxy_pct=100*(imp-exp)/cons,
        official_net_import_reliance_pct=official.get(year,'not_provided_in_inputs'),
        official_reliance_status='reported_exact' if year in official else 'not_provided_in_inputs',
        supply_source='MCS2026',supply_data_status='estimate_flag_not_preserved_in_input',
        price_source='supplied tin_monthly_volatility.csv; benchmark provenance not independently verified'))
write('tin_annual_combined_2021_2025.csv',annual)
by_year={r['year']:r for r in annual}
monthly=[]
for i,r in enumerate(prices):
    if not 2021<=int(r['month'][:4])<=2025:continue
    nextrows=prices[i+1:i+4];assert len(nextrows)==3
    forward=statistics.stdev(returns[x['month']] for x in nextrows)
    assert abs(forward-num(r['next_3_month_volatility_pct']))<1e-5
    monthly.append(dict(month=r['month'],**by_year[int(r['month'][:4])],
        tin_price_usd_per_kg=num(r['tin_price_usd_per_kg']),monthly_price_change_pct=returns[r['month']],
        next_3_month_volatility_pct=forward,forward_window_end=nextrows[-1]['month'],
        annual_fields_usage='descriptive_join_only_not_monthly_flows'))
write('tin_monthly_combined_2021_2025.csv',monthly)
# Context lives at its actual reference period and is not annualized.
context=[]
for r in tin_trade:
    value=num(r['value'])
    if value is None:continue
    context.append(dict(source=r['source'],reporter_country=r['reporter_country'],commodity=r['commodity'],
        period_start=r['period_start'],period_end=r['period_end'],indicator=r['indicator'],
        value=value,unit=r['unit'],status=r['status']))
write('tin_trade_context.csv',context)
print('Annual rows:',len(annual),'Monthly rows:',len(monthly),'Context rows:',len(context))

Annual rows: 5 Monthly rows: 60 Context rows: 8


## Validation, source accounting and first 50 rows
The audit documents the incorrect metric label, the competing ds140 consumption record, missing official dependence years, and excluded broad-product trade. Original CSVs remain unchanged. No official dependence trend can be concluded from one official point. Proxy changes are described separately; the annual summary uses realized volatility, not forward-looking volatility extending into 2026.

In [3]:
assert len(annual)==5 and len(monthly)==60
for table in (annual,monthly,context):
    for r in table:
        assert all(v is not None and str(v).strip() not in ('','nan','NaN') for v in r.values())
        assert all(math.isfinite(v) for v in r.values() if isinstance(v,(int,float)))
for r in annual:
    assert math.isclose(r['refined_net_imports_tonnes'],r['refined_imports_tonnes']-r['refined_exports_tonnes'])
assert official.get(2025)==77
conflicts=[r for r in supply if r['mineral']=='tin' and r['country']=='United States'
           and r['year']=='2021' and r['detail']=='apparent' and r['source']=='ds140']
audit={
 'input_sha256':{name:hashlib.sha256((BASE/'inputs'/name).read_bytes()).hexdigest() for name in ['final_trade_indicators.csv','minerals_supply_demand.csv','tin_monthly_volatility.csv']},
 'annual_rows':len(annual),'monthly_rows':len(monthly),'blank_cells_in_main_tables':0,
 'unavailable_official_reliance_years':[r['year'] for r in annual if r['official_reliance_status']!='reported_exact'],
 'numeric_imputation_count':0,'official_reliance_not_equal_to_proxy':True,
 'annual_proxy_2025_pct':by_year[2025]['net_imports_to_consumption_proxy_pct'],
 'annual_official_2025_pct':official[2025],
 'selection_rules':['Exact tin mineral and exact refined detail; exclude alloys/scrap',
 'Imports selected by detail because metric incorrectly says consumption',
 'MCS2026 consumption preferred consistently; ds140 2021 apparent record not mixed',
 'Broad chapter-26 Comtrade and all non-tin trade excluded',
 'Missing official reliance retained as explicit unavailable label',
 'Pooled 2021–2024 HHI remains in context file only'],
 'competing_2021_ds140_consumption':conflicts,
 'limitations':['Five annual observations cannot establish causality or predictive performance',
 'Price benchmark is not a verified semiconductor packaging price',
 'Forward 3-month metric can extend into 2026 and is not the annual realized volatility measure',
 'Historical revised MCS2026 quantities are descriptive, not point-in-time forecasting features']}
(BASE/'validation_report.json').write_text(json.dumps(audit,indent=2))
from IPython.display import display, HTML
from html import escape
for name in ['tin_annual_combined_2021_2025.csv','tin_monthly_combined_2021_2025.csv','tin_trade_context.csv']:
    with (BASE/name).open(newline='') as f:
        reader=csv.DictReader(f);fields=reader.fieldnames;preview=list(reader)[:50]
    print(name,'— first',len(preview),'rows')
    print(','.join(fields))
    for row in preview:print(','.join(str(row[k]) for k in fields))
    display(HTML('<h4>'+escape(name)+'</h4><div style="overflow:auto;max-height:500px"><table><tr>'+''.join('<th>'+escape(k)+'</th>' for k in fields)+'</tr>'+''.join('<tr>'+''.join('<td>'+escape(str(row[k]))+'</td>' for k in fields)+'</tr>' for row in preview)+'</table></div>'))
print('Year | realized volatility % | net-import/consumption proxy % | official reliance %')
for r in annual:print(r['year'],round(r['realized_monthly_return_volatility_pct'],3),round(r['net_imports_to_consumption_proxy_pct'],3),r['official_net_import_reliance_pct'])

tin_annual_combined_2021_2025.csv — first 5 rows
year,mineral,material_scope,reporter_country,price_month_count,mean_tin_price_usd_per_kg,realized_monthly_return_volatility_pct,refined_imports_tonnes,refined_exports_tonnes,apparent_refined_consumption_tonnes,refined_net_imports_tonnes,net_imports_to_consumption_proxy_pct,official_net_import_reliance_pct,official_reliance_status,supply_source,supply_data_status,price_source
2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified
2022,tin,refined tin,United States,12,31.335333333333335,10.830203302036509,33200.0,1310.0,41200.0,31890.0,77.40291262135922,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not indep

year,mineral,material_scope,reporter_country,price_month_count,mean_tin_price_usd_per_kg,realized_monthly_return_volatility_pct,refined_imports_tonnes,refined_exports_tonnes,apparent_refined_consumption_tonnes,refined_net_imports_tonnes,net_imports_to_consumption_proxy_pct,official_net_import_reliance_pct,official_reliance_status,supply_source,supply_data_status,price_source
2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified
2022,tin,refined tin,United States,12,31.335333333333335,10.830203302036509,33200.0,1310.0,41200.0,31890.0,77.40291262135922,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified
2023,tin,refined tin,United States,12,25.938166666666667,7.67237995928984,28200.0,918.0,35000.0,27282.0,77.94857142857143,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified
2024,tin,refined tin,United States,12,30.066333333333333,5.696928555701936,25400.0,596.0,34600.0,24804.0,71.6878612716763,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified
2025,tin,refined tin,United States,12,34.05883333333333,4.22121581886217,32000.0,850.0,43000.0,31150.0,72.44186046511628,77.0,reported_exact,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified


month,year,mineral,material_scope,reporter_country,price_month_count,mean_tin_price_usd_per_kg,realized_monthly_return_volatility_pct,refined_imports_tonnes,refined_exports_tonnes,apparent_refined_consumption_tonnes,refined_net_imports_tonnes,net_imports_to_consumption_proxy_pct,official_net_import_reliance_pct,official_reliance_status,supply_source,supply_data_status,price_source,tin_price_usd_per_kg,monthly_price_change_pct,next_3_month_volatility_pct,forward_window_end,annual_fields_usage
2021-01-01,2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified,21.92,11.088587066693712,9.455204184217427,2021-04-01,descriptive_join_only_not_monthly_flows
2021-02-01,2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified,26.316,20.05474452554743,5.889599183673618,2021-05-01,descriptive_join_only_not_monthly_flows
2021-03-01,2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified,27.061,2.8309773521811854,6.691727954312012,2021-06-01,descriptive_join_only_not_monthly_flows
2021-04-01,2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified,28.328,4.682014707512661,6.693667276631533,2021-07-01,descriptive_join_only_not_monthly_flows
2021-05-01,2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified,32.246,13.830838746116925,1.9392608689294188,2021-08-01,descriptive_join_only_not_monthly_flows
2021-06-01,2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified,32.503,0.796998077280886,2.572414169307102,2021-09-01,descriptive_join_only_not_monthly_flows
2021-07-01,2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified,34.02,4.667261483555363,4.292957563134286,2021-10-01,descriptive_join_only_not_monthly_flows
2021-08-01,2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified,35.024,2.951205173427396,4.260283159608018,2021-11-01,descriptive_join_only_not_monthly_flows
2021-09-01,2021,tin,refined tin,United States,12,32.384166666666665,6.128355575597468,38100.0,1290.0,48000.0,36810.0,76.6875,not_provided_in_inputs,not_provided_in_inputs,MCS2026,estimate_flag_not_preserved_in_input,supplied tin_monthly_volatility.csv; benchmark provenance not independently verified,34.887,-0.3911603471905001,3.74325705499

source,reporter_country,commodity,period_start,period_end,indicator,value,unit,status
USGS MCS2026 Fig2,United States,"TIN, refined",2025,2025,import_reliance_exact,77.0,percent,exact
USGS MCS2026 Fig2,United States,"TIN, refined",2025,2025,import_reliance_lower_bound,77.0,percent,exact
USGS MCS2026 Fig2,United States,"TIN, refined",2025,2025,import_reliance_upper_bound,77.0,percent,exact
USGS MCS2026 Fig3,United States,"Tin, refined",2021,2024,hhi_lower_bound,0.2015,fraction,bound_from_grouped_shares
USGS MCS2026 Fig3,United States,"Tin, refined",2021,2024,hhi_upper_bound,0.23040000000000002,fraction,bound_from_grouped_shares
USGS MCS2026 Fig3,United States,"Tin, refined",2021,2024,largest_named_supplier_share_pct,31.0,percent,available
USGS MCS2026 Fig3,United States,"Tin, refined",2021,2024,named_supplier_count,4.0,count,reported_named_count_only
USGS MCS2026 Fig3,United States,"Tin, refined",2021,2024,other_share_pct,17.0,percent,available
